In [2]:
# -*- coding: utf-8 -*-
"""
True SHAP interaction heatmaps for representative malodor descriptors

核心目的：
1. 只分析强恶臭相关 odor descriptors；
2. 不分析全部 24 个标签；
3. 使用 SHAP 专门的 shap_interaction_values() 计算两个特征之间的交互；
4. 对代表性结构片段 sulfur groups / amines / aldehydes / carboxylic acids 进行交互热图分析；
5. 输出 feature-level 和 group-level interaction heatmaps；
6. 正值解释为模型层面的协同增强，负值解释为模型层面的拮抗/抑制。

注意：
- 不能直接对全部几千个特征计算 shap_interaction_values，复杂度是 O(N × F²)；
- 因此先用 full model 的普通 SHAP 筛选代表性重要结构特征；
- 然后用这些代表性特征训练 interaction-focused model，计算真实 SHAP interaction values；
- 这部分用于解释“代表性结构片段交互”，不是替代主模型性能评估。
"""

import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")


# ============================================================
# 0. 用户配置
# ============================================================

RANDOM_SEED = 42

FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

OUT_DIR = "./SHAP_true_interaction_malodor_descriptors"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
MODEL_DIR = os.path.join(OUT_DIR, "models")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

for d in [OUT_DIR, PLOT_DIR, TABLE_DIR, MODEL_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

DPI = 600

# 只分析恶臭/刺激性/环境异味相关描述词，不分析 24 个全部标签
MALODOR_LABELS = [
    "sulfurous",
    "garlic",
    "cabbage",
    "fishy",
    "pungent",
    "sharp",
    "sour",
    "cheesy",
    "sweaty",
    "musty"
]

TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

# 每个结构组最多保留多少个代表性特征
TOPK_FEATURES_PER_GROUP = 4

# 每个标签最多用于 interaction 的总特征数
MAX_TOTAL_FEATURES_FOR_INTERACTION = 14

# 计算 interaction values 的最大样本数
# 样本越多越慢；建议 800–1500
MAX_INTERACTION_SAMPLES = 1200

# 如果某个标签中正样本太少，跳过
MIN_POSITIVES = 10

# 是否保存绝对交互强度热图
SAVE_ABS_HEATMAP = True

# 是否使用 interaction-focused reduced model
# 推荐 True；否则 full model 的 interaction matrix 维度太大，容易爆内存。
USE_REDUCED_INTERACTION_MODEL = True


# ============================================================
# 1. XGBoost 参数
# ============================================================

BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)


# ============================================================
# 2. 结构组定义
# ============================================================
# 审稿人明确提到 sulfur groups, amines, aldehydes；
# 对 sour/cheesy/sweaty 等恶臭，也加入 carboxylic acids 作为代表性结构组。

GROUP_PATTERNS = {
    "Sulfur_groups": [
        r"sulfur",
        r"sulphur",
        r"groupscontainingsul",
        r"containing\s*sul",
        r"atom:\s*s",
        r"kg_element_s",
        r"thiol",
        r"thio",
        r"sulfide",
        r"sulfone",
        r"sulfoxide",
        r"sulfanyl",
        r"mercapto",
        r"disulfide",
        r"trisulfide",
        r"thiophene",
        r"\(-sh\)",
        r"n=c=s"
    ],

    "Amines": [
        r"amine",
        r"amines",
        r"amino",
        r"aniline",
        r"ammonia",
        r"ammonium",
        r"imino",
        r"imine",
        r"nitrogen",
        r"groupscontainingnitrogen",
        r"atom:\s*n",
        r"kg_element_n",
        r"pyridine",
        r"pyrrole",
        r"indole",
        r"n\s*≥",
        r"n\s*>=",
        r"n="
    ],

    "Aldehydes_Carbonyls": [
        r"aldehyde",
        r"aldehydic",
        r"formyl",
        r"carbonyl",
        r"c\(=o\)",
        r"\[cx3\]\(=o\)",
        r"\[cx3h1\]\(=o\)",
        r"cc\(=o\)",
        r"ketone",
        r"ketonic",
        r"acrolein",
        r"propanal",
        r"butanal",
        r"hexanal",
        r"benzaldehyde"
    ],

    "Carboxylic_acids": [
        r"carboxylic",
        r"carboxyl",
        r"-cooh",
        r"cooh",
        r"cc\(=o\)o",
        r"c\(=o\)ox2h1",
        r"acid",
        r"fatty",
        r"acetic",
        r"propionic",
        r"butyric",
        r"valeric",
        r"isovaleric"
    ],

    "Ethers_Esters": [
        r"ether",
        r"ester",
        r"ethereal",
        r"acetal",
        r"lactone",
        r"acetate",
        r"c-o-c",
        r"coc"
    ],

    "Aromatics": [
        r"aromatic",
        r"benzene",
        r"phenyl",
        r"toluene",
        r"xylene",
        r"styrene",
        r"heteroaromatic"
    ],
}


# 每个恶臭标签重点分析哪些结构组
# 每个标签只会从这些组中挑选代表性重要特征
LABEL_TO_GROUPS = {
    "sulfurous": ["Sulfur_groups", "Amines", "Aldehydes_Carbonyls"],
    "garlic": ["Sulfur_groups", "Amines", "Aldehydes_Carbonyls"],
    "cabbage": ["Sulfur_groups", "Amines", "Aldehydes_Carbonyls"],

    "fishy": ["Amines", "Sulfur_groups", "Aldehydes_Carbonyls"],

    "pungent": ["Aldehydes_Carbonyls", "Sulfur_groups", "Amines"],
    "sharp": ["Aldehydes_Carbonyls", "Sulfur_groups", "Amines"],

    "sour": ["Carboxylic_acids", "Aldehydes_Carbonyls", "Sulfur_groups"],
    "cheesy": ["Carboxylic_acids", "Sulfur_groups", "Amines"],
    "sweaty": ["Carboxylic_acids", "Sulfur_groups", "Amines"],

    "musty": ["Ethers_Esters", "Aldehydes_Carbonyls", "Aromatics"],
}


# 如果你想手动指定某个标签的代表性特征，填这里。
# 若不为空，则优先使用手动特征；否则自动按 mean(|SHAP|) 选择。
MANUAL_FEATURES_PER_LABEL = {
    # 示例，特征名必须和 Excel 列名完全一致：
    # "sulfurous": [
    #     "Atom: S ≥ 1",
    #     "Atom: S ≥ 2",
    #     "KG_Ancestor_GroupsContainingSulfur",
    #     "Atom: N ≥ 2",
    #     "FG: [CX3H1](=O)",
    # ],
}


# ============================================================
# 3. 数据读取与 X/y 构建
# ============================================================

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None

    priority = [
        "Canonical SMILES", "Canonical_SMILES", "canonical_smiles",
        "SMILES", "smiles", "StdSMILES"
    ]

    for p in priority:
        for c in cand:
            if c.lower() == p.lower():
                return c

    return cand[0]


def is_numeric_or_convertible(series: pd.Series) -> bool:
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def build_X_y(df: pd.DataFrame):
    smiles_col = find_smiles_col(df)

    missing_labels = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if missing_labels:
        raise ValueError(f"缺少标签列: {missing_labels}")

    y_df = df[TARGET_LABELS_24].fillna(0).astype(int)

    exclude = set([c for c in TARGET_LABELS_24 if c in df.columns])
    if smiles_col is not None:
        exclude.add(smiles_col)

    feature_cols = [c for c in df.columns if c not in exclude]

    good_cols = []
    bad_cols = []

    for c in feature_cols:
        if is_numeric_or_convertible(df[c]):
            good_cols.append(c)
        else:
            bad_cols.append(c)

    if bad_cols:
        print(f"[WARN] 删除非数值特征列 {len(bad_cols)} 个：")
        print(bad_cols[:20])

    X_df = df[good_cols].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X_df = X_df.fillna(0).astype(np.float32)

    return X_df, y_df, smiles_col


# ============================================================
# 4. 模型与 SHAP 函数
# ============================================================

def train_xgb_binary(X_values, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    model = xgb.XGBClassifier(**params)
    model.fit(X_values, y_bin)

    return model


def get_shap_values(model, X_values):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_values)

    if isinstance(sv, list):
        if len(sv) == 2:
            sv = sv[1]
        else:
            sv = sv[0]

    return np.asarray(sv, dtype=np.float32)


def get_shap_interaction_values(model, X_values):
    explainer = shap.TreeExplainer(model)
    inter = explainer.shap_interaction_values(X_values)

    if isinstance(inter, list):
        if len(inter) == 2:
            inter = inter[1]
        else:
            inter = inter[0]

    inter = np.asarray(inter, dtype=np.float32)

    if inter.ndim != 3:
        raise ValueError(f"Unexpected SHAP interaction shape: {inter.shape}")

    return inter


# ============================================================
# 5. 特征组匹配与代表特征选择
# ============================================================

def sanitize_filename(s):
    return re.sub(r"[^\w\-_\.]+", "_", str(s))


def match_group(feature_name, group_name):
    f = str(feature_name).lower()

    for pat in GROUP_PATTERNS[group_name]:
        if re.search(pat, f):
            return True

    return False


def assign_groups(feature_name):
    groups = []

    for g in GROUP_PATTERNS:
        if match_group(feature_name, g):
            groups.append(g)

    return groups


def non_constant_feature(X_df, feature):
    vals = X_df[feature].values.astype(float)
    return np.nanstd(vals) > 1e-12


def export_detected_features(feature_names):
    rows = []

    for i, f in enumerate(feature_names):
        groups = assign_groups(f)
        if groups:
            rows.append({
                "feature_index": i,
                "feature": f,
                "matched_groups": ";".join(groups)
            })

    out_df = pd.DataFrame(rows)

    out_path = os.path.join(TABLE_DIR, "detected_typical_substructure_features.csv")
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("[SAVE]", out_path)

    if len(out_df) > 0:
        print("[INFO] Detected typical substructure features:")
        for g in GROUP_PATTERNS:
            n = out_df["matched_groups"].str.contains(g, na=False).sum()
            print(f"  {g}: {n}")

    return out_df


def select_representative_features(label, X_df, full_shap_values, feature_names):
    """
    只为当前恶臭标签选择代表性特征。
    选择逻辑：
    1. 若 MANUAL_FEATURES_PER_LABEL 中有指定，则优先使用；
    2. 否则在 LABEL_TO_GROUPS[label] 指定的结构组中，按当前标签 mean(|SHAP|) 选择 Top K；
    3. 只保留非恒定特征。
    """

    if label in MANUAL_FEATURES_PER_LABEL and len(MANUAL_FEATURES_PER_LABEL[label]) > 0:
        manual = []

        for f in MANUAL_FEATURES_PER_LABEL[label]:
            if f in X_df.columns and non_constant_feature(X_df, f):
                manual.append(f)
            else:
                print(f"[WARN] 手动特征不存在或为常数，跳过：{f}")

        if len(manual) >= 2:
            rows = []

            mean_abs = np.mean(np.abs(full_shap_values), axis=0)

            for f in manual:
                idx = feature_names.index(f)
                groups = assign_groups(f)
                rows.append({
                    "label": label,
                    "group": groups[0] if groups else "Manual",
                    "feature_index": idx,
                    "feature": f,
                    "mean_abs_SHAP": float(mean_abs[idx]),
                    "selection_mode": "manual"
                })

            return pd.DataFrame(rows)

    groups_for_label = LABEL_TO_GROUPS.get(label, ["Sulfur_groups", "Amines", "Aldehydes_Carbonyls"])

    mean_abs = np.mean(np.abs(full_shap_values), axis=0)

    selected_rows = []

    for group in groups_for_label:
        candidates = []

        for idx, f in enumerate(feature_names):
            if not match_group(f, group):
                continue

            if not non_constant_feature(X_df, f):
                continue

            candidates.append({
                "label": label,
                "group": group,
                "feature_index": idx,
                "feature": f,
                "mean_abs_SHAP": float(mean_abs[idx]),
                "selection_mode": "auto_by_group_shap"
            })

        candidates = sorted(candidates, key=lambda r: r["mean_abs_SHAP"], reverse=True)
        selected_rows.extend(candidates[:TOPK_FEATURES_PER_GROUP])

    selected_df = pd.DataFrame(selected_rows)

    if selected_df.empty:
        return selected_df

    selected_df = selected_df.sort_values("mean_abs_SHAP", ascending=False)

    if len(selected_df) > MAX_TOTAL_FEATURES_FOR_INTERACTION:
        selected_df = selected_df.head(MAX_TOTAL_FEATURES_FOR_INTERACTION)

    selected_df = selected_df.sort_values(["group", "mean_abs_SHAP"], ascending=[True, False])
    selected_df = selected_df.reset_index(drop=True)

    return selected_df


def stratified_sample_indices(y_bin, max_n=1200, random_seed=42):
    """
    分层采样用于 SHAP interaction 计算。
    优先保留正样本，再抽取负样本。
    """

    y_bin = np.asarray(y_bin)
    n = len(y_bin)

    if n <= max_n:
        return np.arange(n)

    rng = np.random.default_rng(random_seed)

    pos_idx = np.where(y_bin == 1)[0]
    neg_idx = np.where(y_bin == 0)[0]

    max_pos = min(len(pos_idx), max_n // 2)

    if len(pos_idx) <= max_pos:
        chosen_pos = pos_idx
    else:
        chosen_pos = rng.choice(pos_idx, size=max_pos, replace=False)

    remaining = max_n - len(chosen_pos)

    if len(neg_idx) <= remaining:
        chosen_neg = neg_idx
    else:
        chosen_neg = rng.choice(neg_idx, size=remaining, replace=False)

    chosen = np.concatenate([chosen_pos, chosen_neg])
    rng.shuffle(chosen)

    return chosen


# ============================================================
# 6. 交互矩阵计算与热图
# ============================================================

def short_group_name(group):
    mapping = {
        "Sulfur_groups": "S",
        "Amines": "N",
        "Aldehydes_Carbonyls": "CHO",
        "Carboxylic_acids": "COOH",
        "Ethers_Esters": "Ether/Ester",
        "Aromatics": "Aro"
    }
    return mapping.get(group, group)


def build_display_names(selected_df):
    names = []

    for _, row in selected_df.iterrows():
        names.append(f"[{short_group_name(row['group'])}] {row['feature']}")

    return names


def zero_diagonal(mat):
    mat2 = mat.copy()
    np.fill_diagonal(mat2, 0.0)
    return mat2


def plot_signed_heatmap(matrix, labels, out_png, title, cbar_label):
    mat = np.asarray(matrix, dtype=float)

    fig_w = max(8.0, 0.55 * len(labels))
    fig_h = max(6.5, 0.55 * len(labels))

    plt.figure(figsize=(fig_w, fig_h))

    vmax = np.nanmax(np.abs(mat))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = plt.imshow(mat, cmap="bwr", vmin=-vmax, vmax=vmax, aspect="auto")

    plt.colorbar(im, label=cbar_label)

    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=8)
    plt.yticks(np.arange(len(labels)), labels, fontsize=8)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                plt.text(
                    j, i,
                    f"{val:+.3f}",
                    ha="center",
                    va="center",
                    fontsize=6
                )

    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


def plot_abs_heatmap(matrix, labels, out_png, title, cbar_label):
    mat = np.asarray(matrix, dtype=float)

    fig_w = max(8.0, 0.55 * len(labels))
    fig_h = max(6.5, 0.55 * len(labels))

    plt.figure(figsize=(fig_w, fig_h))

    vmax = np.nanmax(mat)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = plt.imshow(mat, cmap="viridis", vmin=0, vmax=vmax, aspect="auto")

    plt.colorbar(im, label=cbar_label)

    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=8)
    plt.yticks(np.arange(len(labels)), labels, fontsize=8)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                plt.text(
                    j, i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=6
                )

    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


def summarize_pairwise_interactions(label, mean_inter, mean_abs_inter, selected_features, selected_groups):
    rows = []
    m = len(selected_features)

    for i in range(m):
        for j in range(i + 1, m):
            signed_val = float(mean_inter[i, j])
            abs_val = float(mean_abs_inter[i, j])

            if signed_val > 0:
                direction = "synergistic_positive"
            elif signed_val < 0:
                direction = "antagonistic_negative"
            else:
                direction = "near_zero"

            rows.append({
                "label": label,
                "feature_i": selected_features[i],
                "group_i": selected_groups[i],
                "feature_j": selected_features[j],
                "group_j": selected_groups[j],
                "signed_mean_SHAP_interaction": signed_val,
                "mean_abs_SHAP_interaction": abs_val,
                "interaction_direction": direction
            })

    out_df = pd.DataFrame(rows)

    if not out_df.empty:
        out_df = out_df.sort_values("mean_abs_SHAP_interaction", ascending=False)

    out_path = os.path.join(TABLE_DIR, f"pairwise_SHAP_interaction_summary__{label}.csv")
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("[SAVE]", out_path)

    return out_df


def aggregate_group_interactions(label, mean_inter, mean_abs_inter, selected_groups):
    groups = list(dict.fromkeys(selected_groups))

    group_to_idx = {g: [] for g in groups}

    for i, g in enumerate(selected_groups):
        group_to_idx[g].append(i)

    signed_mat = np.full((len(groups), len(groups)), np.nan)
    abs_mat = np.full((len(groups), len(groups)), np.nan)
    n_mat = np.zeros((len(groups), len(groups)), dtype=int)

    for a, ga in enumerate(groups):
        idx_a = group_to_idx[ga]

        for b, gb in enumerate(groups):
            idx_b = group_to_idx[gb]

            if len(idx_a) == 0 or len(idx_b) == 0:
                continue

            sub_signed = mean_inter[np.ix_(idx_a, idx_b)]
            sub_abs = mean_abs_inter[np.ix_(idx_a, idx_b)]

            if ga == gb:
                if len(idx_a) <= 1:
                    continue

                mask = ~np.eye(len(idx_a), dtype=bool)
                signed_vals = sub_signed[mask]
                abs_vals = sub_abs[mask]
            else:
                signed_vals = sub_signed.flatten()
                abs_vals = sub_abs.flatten()

            signed_mat[a, b] = float(np.mean(signed_vals))
            abs_mat[a, b] = float(np.mean(abs_vals))
            n_mat[a, b] = int(len(signed_vals))

    short_labels = [short_group_name(g) for g in groups]

    pd.DataFrame(signed_mat, index=groups, columns=groups).to_csv(
        os.path.join(TABLE_DIR, f"group_signed_interaction_matrix__{label}.csv"),
        encoding="utf-8-sig"
    )

    pd.DataFrame(abs_mat, index=groups, columns=groups).to_csv(
        os.path.join(TABLE_DIR, f"group_abs_interaction_matrix__{label}.csv"),
        encoding="utf-8-sig"
    )

    rows = []

    for i, gi in enumerate(groups):
        for j, gj in enumerate(groups):
            val = signed_mat[i, j]
            aval = abs_mat[i, j]

            if not np.isfinite(val):
                continue

            if val > 0:
                direction = "synergistic_positive"
            elif val < 0:
                direction = "antagonistic_negative"
            else:
                direction = "near_zero"

            rows.append({
                "label": label,
                "group_i": gi,
                "group_j": gj,
                "signed_mean_SHAP_interaction": float(val),
                "mean_abs_SHAP_interaction": float(aval),
                "n_feature_pairs": int(n_mat[i, j]),
                "interaction_direction": direction
            })

    group_long = pd.DataFrame(rows)

    group_long.to_csv(
        os.path.join(TABLE_DIR, f"group_interaction_summary__{label}.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    return signed_mat, abs_mat, groups, short_labels, group_long


def plot_group_heatmap(matrix, groups, short_labels, out_png, title, signed=True):
    mat = np.asarray(matrix, dtype=float)

    plt.figure(figsize=(6.2, 5.2))

    if signed:
        vmax = np.nanmax(np.abs(mat))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        im = plt.imshow(mat, cmap="bwr", vmin=-vmax, vmax=vmax, aspect="auto")
        cbar_label = "Mean SHAP interaction value"
    else:
        vmax = np.nanmax(mat)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        im = plt.imshow(mat, cmap="viridis", vmin=0, vmax=vmax, aspect="auto")
        cbar_label = "Mean |SHAP interaction value|"

    plt.colorbar(im, label=cbar_label)

    plt.xticks(np.arange(len(short_labels)), short_labels, rotation=45, ha="right", fontsize=10)
    plt.yticks(np.arange(len(short_labels)), short_labels, fontsize=10)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                plt.text(
                    j, i,
                    f"{val:+.3f}" if signed else f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=9
                )

    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    print("[SAVE]", out_png)


# ============================================================
# 7. 单个恶臭标签的完整交互分析
# ============================================================

def run_label_interaction(label, X_df, y_df, feature_names):
    print("\n" + "=" * 100)
    print(f"[LABEL] {label}")
    print("=" * 100)

    y_bin = y_df[label].values.astype(int)

    n_pos = int(y_bin.sum())
    n_neg = int(len(y_bin) - n_pos)

    print(f"[INFO] positives={n_pos}, negatives={n_neg}")

    if n_pos < MIN_POSITIVES:
        print(f"[WARN] {label} positive samples < {MIN_POSITIVES}, skipped.")
        return None

    # ------------------------------------------------------------
    # Step 1. Full model + normal SHAP，用于筛选代表性重要结构特征
    # ------------------------------------------------------------
    print("[INFO] Training full model for feature screening...")
    full_model = train_xgb_binary(X_df.values, y_bin)

    full_model_path = os.path.join(MODEL_DIR, f"xgb_full__{label}.json")
    full_model.get_booster().save_model(full_model_path)

    print("[INFO] Calculating normal SHAP values for feature screening...")
    full_sv = get_shap_values(full_model, X_df.values)

    selected_df = select_representative_features(
        label=label,
        X_df=X_df,
        full_shap_values=full_sv,
        feature_names=feature_names
    )

    if selected_df.empty or len(selected_df) < 2:
        print(f"[WARN] Not enough representative features for {label}.")
        return None

    selected_features = selected_df["feature"].tolist()
    selected_groups = selected_df["group"].tolist()

    selected_path = os.path.join(TABLE_DIR, f"selected_representative_features__{label}.csv")
    selected_df.to_csv(selected_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", selected_path)

    print("[INFO] Selected representative features:")
    print(selected_df[["group", "feature", "mean_abs_SHAP"]].to_string(index=False))

    # ------------------------------------------------------------
    # Step 2. Interaction-focused model
    # ------------------------------------------------------------
    X_inter_df = X_df[selected_features].copy()

    print(f"[INFO] Training interaction-focused model with {X_inter_df.shape[1]} representative features...")

    if USE_REDUCED_INTERACTION_MODEL:
        inter_model = train_xgb_binary(X_inter_df.values, y_bin)
        X_for_interaction = X_inter_df.values
    else:
        raise RuntimeError(
            "Full-model interaction values are not recommended because F^2 is too large. "
            "Set USE_REDUCED_INTERACTION_MODEL=True."
        )

    inter_model_path = os.path.join(MODEL_DIR, f"xgb_interaction_focused__{label}.json")
    inter_model.get_booster().save_model(inter_model_path)

    # ------------------------------------------------------------
    # Step 3. Sampling
    # ------------------------------------------------------------
    sample_idx = stratified_sample_indices(
        y_bin,
        max_n=MAX_INTERACTION_SAMPLES,
        random_seed=RANDOM_SEED
    )

    X_sample = X_for_interaction[sample_idx]
    y_sample = y_bin[sample_idx]

    print(f"[INFO] SHAP interaction samples={len(sample_idx)}, positives={int(y_sample.sum())}")

    # ------------------------------------------------------------
    # Step 4. True SHAP interaction values
    # ------------------------------------------------------------
    print("[INFO] Calculating true SHAP interaction values...")
    inter_values = get_shap_interaction_values(inter_model, X_sample)

    # n_sample × n_feature × n_feature
    mean_inter = np.mean(inter_values, axis=0)
    mean_abs_inter = np.mean(np.abs(inter_values), axis=0)

    # diagonal 是主效应，不是两个特征交互；热图中置 0
    mean_inter_no_diag = zero_diagonal(mean_inter)
    mean_abs_inter_no_diag = zero_diagonal(mean_abs_inter)

    np.savez_compressed(
        os.path.join(CACHE_DIR, f"shap_interaction_values__{label}.npz"),
        interaction_values=inter_values.astype(np.float16),
        selected_features=np.array(selected_features),
        selected_groups=np.array(selected_groups),
        sample_indices=sample_idx
    )

    # ------------------------------------------------------------
    # Step 5. Feature-level heatmap
    # ------------------------------------------------------------
    display_names = build_display_names(selected_df)

    signed_feature_png = os.path.join(
        PLOT_DIR,
        f"FeatureLevel_signed_SHAP_interaction_heatmap__{label}.png"
    )

    plot_signed_heatmap(
        mean_inter_no_diag,
        display_names,
        signed_feature_png,
        title=f"{label}: feature-level SHAP interaction heatmap",
        cbar_label="Mean SHAP interaction value"
    )

    if SAVE_ABS_HEATMAP:
        abs_feature_png = os.path.join(
            PLOT_DIR,
            f"FeatureLevel_abs_SHAP_interaction_heatmap__{label}.png"
        )

        plot_abs_heatmap(
            mean_abs_inter_no_diag,
            display_names,
            abs_feature_png,
            title=f"{label}: feature-level interaction strength",
            cbar_label="Mean |SHAP interaction value|"
        )

    pd.DataFrame(
        mean_inter_no_diag,
        index=display_names,
        columns=display_names
    ).to_csv(
        os.path.join(TABLE_DIR, f"feature_signed_interaction_matrix__{label}.csv"),
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        mean_abs_inter_no_diag,
        index=display_names,
        columns=display_names
    ).to_csv(
        os.path.join(TABLE_DIR, f"feature_abs_interaction_matrix__{label}.csv"),
        encoding="utf-8-sig"
    )

    pair_df = summarize_pairwise_interactions(
        label,
        mean_inter_no_diag,
        mean_abs_inter_no_diag,
        selected_features,
        selected_groups
    )

    # ------------------------------------------------------------
    # Step 6. Group-level heatmap
    # ------------------------------------------------------------
    signed_group_mat, abs_group_mat, group_names, short_labels, group_long = aggregate_group_interactions(
        label,
        mean_inter_no_diag,
        mean_abs_inter_no_diag,
        selected_groups
    )

    signed_group_png = os.path.join(
        PLOT_DIR,
        f"GroupLevel_signed_SHAP_interaction_heatmap__{label}.png"
    )

    plot_group_heatmap(
        signed_group_mat,
        group_names,
        short_labels,
        signed_group_png,
        title=f"{label}: group-level SHAP interaction heatmap",
        signed=True
    )

    if SAVE_ABS_HEATMAP:
        abs_group_png = os.path.join(
            PLOT_DIR,
            f"GroupLevel_abs_SHAP_interaction_heatmap__{label}.png"
        )

        plot_group_heatmap(
            abs_group_mat,
            group_names,
            short_labels,
            abs_group_png,
            title=f"{label}: group-level interaction strength",
            signed=False
        )

    return {
        "label": label,
        "selected_features": selected_df,
        "pairwise": pair_df,
        "group": group_long
    }


# ============================================================
# 8. 主程序
# ============================================================

def main():
    print("[INFO] Reading feature file:")
    print(FEATURE_FILE)

    df = pd.read_excel(FEATURE_FILE)

    X_df, y_df, smiles_col = build_X_y(df)
    feature_names = X_df.columns.tolist()

    print(f"[INFO] X shape = {X_df.shape}")
    print(f"[INFO] y shape = {y_df.shape}")
    print(f"[INFO] n_features = {len(feature_names)}")

    if smiles_col is not None:
        print("[INFO] SMILES column:", smiles_col)

    # 输出所有可识别的典型亚结构特征，方便检查
    export_detected_features(feature_names)

    meta = {
        "feature_file": FEATURE_FILE,
        "malodor_labels": MALODOR_LABELS,
        "group_patterns": GROUP_PATTERNS,
        "label_to_groups": LABEL_TO_GROUPS,
        "topk_features_per_group": TOPK_FEATURES_PER_GROUP,
        "max_total_features_for_interaction": MAX_TOTAL_FEATURES_FOR_INTERACTION,
        "max_interaction_samples": MAX_INTERACTION_SAMPLES,
        "use_reduced_interaction_model": USE_REDUCED_INTERACTION_MODEL,
        "interpretation": {
            "positive_mean_SHAP_interaction": "model-level synergistic effect",
            "negative_mean_SHAP_interaction": "model-level antagonistic or suppressive effect"
        }
    }

    with open(os.path.join(OUT_DIR, "analysis_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    all_selected = []
    all_pairwise = []
    all_group = []

    for label in MALODOR_LABELS:
        if label not in y_df.columns:
            print(f"[WARN] {label} not found in label columns, skipped.")
            continue

        result = run_label_interaction(
            label=label,
            X_df=X_df,
            y_df=y_df,
            feature_names=feature_names
        )

        if result is None:
            continue

        all_selected.append(result["selected_features"])
        all_pairwise.append(result["pairwise"])
        all_group.append(result["group"])

    if all_selected:
        df_selected = pd.concat(all_selected, axis=0, ignore_index=True)
        df_selected.to_excel(
            os.path.join(TABLE_DIR, "ALL_selected_representative_features.xlsx"),
            index=False
        )
        df_selected.to_csv(
            os.path.join(TABLE_DIR, "ALL_selected_representative_features.csv"),
            index=False,
            encoding="utf-8-sig"
        )

    if all_pairwise:
        df_pairwise = pd.concat(all_pairwise, axis=0, ignore_index=True)
        df_pairwise.to_excel(
            os.path.join(TABLE_DIR, "ALL_pairwise_SHAP_interaction_summary.xlsx"),
            index=False
        )
        df_pairwise.to_csv(
            os.path.join(TABLE_DIR, "ALL_pairwise_SHAP_interaction_summary.csv"),
            index=False,
            encoding="utf-8-sig"
        )

        top20 = (
            df_pairwise
            .sort_values(["label", "mean_abs_SHAP_interaction"], ascending=[True, False])
            .groupby("label")
            .head(20)
            .reset_index(drop=True)
        )

        top20.to_excel(
            os.path.join(TABLE_DIR, "ALL_top20_pairwise_interactions_by_label.xlsx"),
            index=False
        )

    if all_group:
        df_group = pd.concat(all_group, axis=0, ignore_index=True)
        df_group.to_excel(
            os.path.join(TABLE_DIR, "ALL_group_level_SHAP_interaction_summary.xlsx"),
            index=False
        )
        df_group.to_csv(
            os.path.join(TABLE_DIR, "ALL_group_level_SHAP_interaction_summary.csv"),
            index=False,
            encoding="utf-8-sig"
        )

    print("\n[DONE] SHAP interaction heatmap analysis finished.")
    print("Outputs:")
    print(" - Feature-level heatmaps:", PLOT_DIR)
    print(" - Group-level heatmaps:", PLOT_DIR)
    print(" - Tables:", TABLE_DIR)


if __name__ == "__main__":
    main()

[INFO] Reading feature file:
./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X shape = (3756, 2595)
[INFO] y shape = (3756, 24)
[INFO] n_features = 2595
[INFO] SMILES column: Canonical_SMILES
[SAVE] ./SHAP_true_interaction_malodor_descriptors/tables/detected_typical_substructure_features.csv
[INFO] Detected typical substructure features:
  Sulfur_groups: 67
  Amines: 50
  Aldehydes_Carbonyls: 41
  Carboxylic_acids: 31
  Ethers_Esters: 57
  Aromatics: 35

[LABEL] sulfurous
[INFO] positives=398, negatives=3358
[INFO] Training full model for feature screening...
[INFO] Calculating normal SHAP values for feature screening...
[SAVE] ./SHAP_true_interaction_malodor_descriptors/tables/selected_representative_features__sulfurous.csv
[INFO] Selected representative features:
              group                             feature  mean_abs_SHAP
Aldehydes_Carbonyls               FG: [CX3](=O)[#6][#6]       0.016126
Aldehydes_Carbonyls                C(=O)O[#6] count >=2       0.009853
Ald